# Case Study: A Week in the Data Team at ShopEasy

**One story. Three chapters. All the concepts from this module.**

---

This notebook follows **Maya**, a junior data analyst who has just joined ShopEasy — a Singapore-based online retailer. Over the course of one week, Maya is given three real data problems by her manager. Each problem maps to one part of the lesson:

| Chapter | Maya's task | Concepts covered |
|---|---|---|
| **1 — The Monday Problem** | Is our sales data trustworthy? | Law of Large Numbers · Mean · Median · Mode |
| **2 — The Wednesday Problem** | What does our customer data actually look like? | Distributions · Normal · Skewed · CLT |
| **3 — The Friday Problem** | Did our new checkout page actually work? | Z-scores · P-values · Hypothesis testing |

**How to use this notebook:** Read every markdown cell before running the code. Each chapter ends with a reflection — write your answers before moving on. The goal is not just to run the code, but to think like Maya.

In [ ]:
# Run this first — loads everything the notebook needs
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('✅ Ready. Let\'s follow Maya through her week.')

---
# Chapter 1 — The Monday Problem
## "Can we trust this number?"

It's Maya's first Monday at ShopEasy. Her manager, Wei, drops by her desk:

> *"We ran a flash sale on Saturday. I checked the dashboard this morning and it says our average order value was $183 — that seems really high. We normally see around $95. Before I put this in the board report, I need you to tell me: is that number real, or is it noise from a small sample?"*

Maya opens the sales database. The flash sale ran for 4 hours on Saturday morning and captured **47 orders**. The rest of the week had hundreds of orders per day.

She thinks: *"47 orders. Is that enough to trust a $183 average?"*

This is the **Law of Large Numbers** problem. Maya needs to understand how sample size affects reliability before she can answer Wei.

### ⏸️ Pause and Predict

Before running the next cell, think about this:

- If the true average order value is $95, could 47 random orders produce an average of $183 just by chance?
- Would your answer change if there were 4,700 orders instead of 47?

*Write your prediction here before running the cell.*

In [ ]:
# Maya runs a simulation to understand the effect of sample size.
# She simulates drawing samples of different sizes from a population
# where the true average order value is $95.

true_mean = 95       # the "real" average order value across all customers
true_std  = 40       # typical variation in order values
sample_sizes = [5, 10, 20, 47, 100, 500, 1000, 5000]

print(f'True average order value: ${true_mean}')
print(f'\nSimulated sample averages:')
print(f'{"Sample size":>12}  {"Observed avg":>14}  {"Difference from true":>20}')
print('-' * 52)
for n in sample_sizes:
    sample = np.random.normal(true_mean, true_std, n).clip(10, None)
    obs_mean = np.mean(sample)
    diff = obs_mean - true_mean
    flag = '  ← like Maya\'s 47 orders' if n == 47 else ''
    print(f'{n:>12}  ${obs_mean:>12.2f}  {diff:>+19.2f}{flag}')

In [ ]:
# Visualise how the observed average converges to the true value as n grows
observed_means = []
for n in range(1, 500):
    sample = np.random.normal(true_mean, true_std, n).clip(10, None)
    observed_means.append(np.mean(sample))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(range(1, 500), observed_means, color='steelblue', linewidth=1.2, alpha=0.8,
        label='Observed sample average')
ax.axhline(true_mean, color='orange', linewidth=2, linestyle='--',
           label=f'True average (${true_mean})')
ax.axvline(47, color='coral', linewidth=1.5, linestyle=':', label='Maya\'s sample (n=47)')
ax.set_xlabel('Number of orders in sample')
ax.set_ylabel('Average order value ($)')
ax.set_title('Law of Large Numbers — Sample Average Converges to True Average')
ax.legend()
ax.set_xlim(1, 500)
plt.tight_layout()
plt.show()

### 💡 What Maya notices

The orange dashed line is the *true* average — $95. With tiny samples (n < 20), the observed average bounces wildly. At n=47, Maya's sample, it's still quite noisy. At n=500+, it has settled very close to the true value.

Maya's conclusion: **the $183 figure from 47 flash sale orders cannot be trusted as a reliable estimate of the true average order value.** It is likely an upward fluke from a small, unusual sample — possibly because the flash sale attracted high-intent buyers who were already planning large purchases.

She tells Wei: *"The 47 orders aren't enough. We'd need at least a few hundred before I'd be confident that $183 reflects a real shift, not just lucky sampling."*

Wei nods: *"Good catch. Let's keep it out of the board report for now."*

### Describing the data that IS reliable

For the weekly summary, Maya uses the full week of order data — 847 orders. She needs to pick the right "average" to report.

**Before running:** Which measure of centre do you think will best represent a "typical" ShopEasy order?

In [ ]:
# Full week of order data — 847 orders
# Most customers buy everyday items; a small number buy premium electronics
np.random.seed(7)
everyday_orders   = np.random.normal(75, 20, 720).clip(15, None)    # 85% of orders
premium_orders    = np.random.normal(380, 80, 127).clip(200, None)  # 15% high-value orders
weekly_orders = np.concatenate([everyday_orders, premium_orders])
np.random.shuffle(weekly_orders)

mean_val   = np.mean(weekly_orders)
median_val = np.median(weekly_orders)
mode_val   = stats.mode(weekly_orders.astype(int))[0]

print(f'Weekly order data ({len(weekly_orders)} orders)')
print(f'  Mean:   ${mean_val:.2f}  ← pulled up by premium orders')
print(f'  Median: ${median_val:.2f}  ← better reflects a typical customer')
print(f'  Mode:   ${mode_val:.2f}')

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(weekly_orders, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(mean_val,   color='orange', linewidth=2.5, label=f'Mean   ${mean_val:.0f}')
ax.axvline(median_val, color='green',  linewidth=2.5, linestyle='--', label=f'Median ${median_val:.0f}')
ax.set_xlabel('Order Value ($)')
ax.set_ylabel('Number of Orders')
ax.set_title('ShopEasy Weekly Order Distribution — Mean vs Median')
ax.legend()
plt.tight_layout()
plt.show()

### Chapter 1 Reflection

1. The mean order value is higher than the median. What does the shape of the histogram tell you about why?
2. If Maya reports the *mean* to Wei, is she being misleading? Which measure should go in the weekly report, and why?
3. In your own words, explain to someone who has never studied statistics why 47 orders wasn't enough for the flash sale analysis.

*Double-click to write your answers here.*

---
# Chapter 2 — The Wednesday Problem
## "What does our customer data actually look like?"

It's Wednesday. The head of marketing, Priya, has asked Maya's team to build a customer segmentation model. Before any modelling can begin, Maya needs to understand the *shape* of ShopEasy's customer data.

Priya hands her three datasets:

1. **Session length** — how many minutes customers spend on the site per visit
2. **Orders per month** — how many orders each customer places per month
3. **Daily new signups** — the number of new accounts created each day

> *"Before we feed this into any model,"* Maya's senior colleague tells her, *"you need to know what distribution each variable follows. The model makes different assumptions depending on the shape. Getting this wrong is how you end up with a model that looks great in testing and fails in production."*

### ⏸️ Pause and Predict

Before running, predict the shape of each variable:
- **Session length:** Most people browse briefly; a few spend a long time. What shape?
- **Orders per month:** Most customers order rarely; loyal ones order often. What shape?
- **Daily signups:** Random day-to-day variation around a steady average. What shape?

*Write your predictions here.*

In [ ]:
# Generate ShopEasy customer data
np.random.seed(42)

# Session length: most sessions are short (right-skewed)
session_lengths = np.random.exponential(scale=8, size=2000).clip(0.5, 120)

# Orders per month: most customers order 0-2 times; power users order more (right-skewed)
orders_per_month = np.random.exponential(scale=1.8, size=2000).clip(0, 20)

# Daily signups: random variation around a daily mean (approximately normal)
daily_signups = np.random.normal(loc=340, scale=45, size=365).clip(100, None).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(session_lengths, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Session Length (minutes)')
axes[0].set_xlabel('Minutes')

axes[1].hist(orders_per_month, bins=30, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_title('Orders per Month')
axes[1].set_xlabel('Orders')

axes[2].hist(daily_signups, bins=30, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[2].set_title('Daily New Signups')
axes[2].set_xlabel('Signups')

for ax in axes:
    ax.set_ylabel('Frequency')
plt.suptitle('ShopEasy Customer Data — Shape of Three Variables', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 💡 What Maya sees

- **Session length and orders per month** are both **right-skewed** — a long tail of power users extends to the right. The mean would overstate the typical customer.
- **Daily signups** looks roughly **normal** — random variation around a steady mean of ~340.

This matters for modelling. Many algorithms assume features are roughly normally distributed. Before feeding session length into a model, Maya would likely apply a **log transformation** to compress the skew.

In [ ]:
# Maya demonstrates why the log transformation helps
log_sessions = np.log1p(session_lengths)   # log1p = log(x+1), handles zero safely

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(session_lengths, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Session Length — Original (right-skewed)')
axes[0].set_xlabel('Minutes')

axes[1].hist(log_sessions, bins=40, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[1].set_title('Session Length — After Log Transform (more symmetric)')
axes[1].set_xlabel('log(Minutes + 1)')

plt.tight_layout()
plt.show()
print('After log transform, the distribution is much closer to normal.')
print('This is what you would feed into a model that assumes normality.')

### The Central Limit Theorem saves the day

Even with skewed raw data, the **Central Limit Theorem** means that if Maya is working with *averages* across groups of customers, those averages will behave normally — regardless of the underlying skew.

Maya's manager asks: *"What's the average session length for customers who buy vs customers who just browse?"* Because she's comparing group averages (not individual sessions), the CLT applies.

In [ ]:
# Compare average session lengths: buyers vs browsers
np.random.seed(42)
buyers   = np.random.exponential(scale=12, size=500).clip(0.5, 120)   # buyers linger longer
browsers = np.random.exponential(scale=6,  size=1500).clip(0.5, 120)  # browsers are quicker

# Show that sample means follow a normal distribution (CLT in action)
buyer_means   = [np.mean(np.random.choice(buyers,   50)) for _ in range(1000)]
browser_means = [np.mean(np.random.choice(browsers, 50)) for _ in range(1000)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw data — skewed
axes[0].hist(buyers,   bins=30, alpha=0.6, color='steelblue',      label='Buyers',   edgecolor='white')
axes[0].hist(browsers, bins=30, alpha=0.6, color='coral', label='Browsers', edgecolor='white')
axes[0].set_title('Raw Session Lengths (skewed)')
axes[0].set_xlabel('Minutes')
axes[0].legend()

# Sample means — normal (CLT)
axes[1].hist(buyer_means,   bins=30, alpha=0.6, color='steelblue',      label='Buyer sample means',   edgecolor='white')
axes[1].hist(browser_means, bins=30, alpha=0.6, color='coral', label='Browser sample means', edgecolor='white')
axes[1].set_title('Distribution of Sample Means (CLT — now normal!)')
axes[1].set_xlabel('Mean session length (minutes)')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Buyers   — mean session: {np.mean(buyers):.1f} min')
print(f'Browsers — mean session: {np.mean(browsers):.1f} min')

### Chapter 2 Reflection

1. Look at the two charts above. The raw session lengths are skewed, but the sample means form a bell curve. In your own words, explain why this happens.
2. Maya wants to use session length as a feature in a machine learning model that assumes normally distributed inputs. What should she do to the data first, and why?
3. The daily signups data was roughly normal even before any transformation. What real-world reason might explain why the number of new signups each day follows a normal distribution?

*Double-click to write your answers here.*

---
# Chapter 3 — The Friday Problem
## "Did it actually work?"

It's Friday afternoon. The product team has been running an experiment for three weeks: a redesigned checkout page (Version B) vs the original (Version A). 600 customers were randomly shown Version A; 600 were shown Version B.

The product manager, James, walks over to Maya's desk:

> *"Version B has a 6.8% conversion rate. Version A has 5.3%. That's a 1.5 percentage point improvement. The design team is ready to ship it. But I need you to tell me — is this a real improvement, or did we just get lucky with who was assigned to each group?"*

Maya knows this is a **hypothesis testing** problem. Before she can answer James, she needs to set up the test properly.

### Step 1 — State the hypotheses

Before touching the data, Maya writes down her hypotheses:

> **H₀ (null hypothesis):** Version B has the same conversion rate as Version A. Any observed difference is due to random chance.

> **H₁ (alternative hypothesis):** Version B has a genuinely higher conversion rate than Version A.

**The significance threshold:** Maya uses α = 0.05. This means she is willing to accept a 5% chance of concluding Version B is better when it actually isn't (a false positive).

In [ ]:
# Simulate the A/B test data
# True conversion rates: A = 5.3%, B = 6.8%  (as James reported)
np.random.seed(42)
n = 600
conversions_a = np.random.binomial(1, 0.053, n)   # Version A: 5.3% true rate
conversions_b = np.random.binomial(1, 0.068, n)   # Version B: 6.8% true rate

rate_a = conversions_a.mean()
rate_b = conversions_b.mean()

print('A/B Test Results')
print(f'  Version A: {conversions_a.sum()} conversions out of {n} → {rate_a:.1%}')
print(f'  Version B: {conversions_b.sum()} conversions out of {n} → {rate_b:.1%}')
print(f'  Observed difference: {(rate_b - rate_a)*100:+.1f} percentage points')

### Step 2 — Visualise before testing

Maya always plots the data before running any test. A good visualisation often reveals something the numbers alone miss.

**Before running:** What do you expect the chart to show?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: conversion rate comparison
categories = ['Did not convert', 'Converted']
a_counts = [(1 - rate_a) * 100, rate_a * 100]
b_counts = [(1 - rate_b) * 100, rate_b * 100]

x = np.arange(len(categories))
width = 0.35
axes[0].bar(x - width/2, a_counts, width, label='Version A', color='steelblue', edgecolor='white')
axes[0].bar(x + width/2, b_counts, width, label='Version B', color='coral',     edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories)
axes[0].set_ylabel('Percentage of Users (%)')
axes[0].set_title('Conversion: Version A vs Version B')
axes[0].legend()

# Right: sampling distribution under the null hypothesis
# If H₀ is true (no real difference), what range of differences would we expect by chance?
null_diffs = []
combined_rate = (conversions_a.sum() + conversions_b.sum()) / (2 * n)
for _ in range(10000):
    sim_a = np.random.binomial(1, combined_rate, n).mean()
    sim_b = np.random.binomial(1, combined_rate, n).mean()
    null_diffs.append(sim_b - sim_a)

observed_diff = rate_b - rate_a
axes[1].hist(null_diffs, bins=50, color='lightsteelblue', edgecolor='white', alpha=0.85,
             label='Differences if H₀ were true')
axes[1].axvline(observed_diff, color='coral', linewidth=2.5,
                label=f'Observed difference ({observed_diff*100:+.1f}pp)')
axes[1].set_xlabel('Difference in conversion rate (B minus A)')
axes[1].set_title('Null Distribution vs Observed Difference')
axes[1].legend()

plt.tight_layout()
plt.show()

### 💡 Reading the null distribution chart

The blue histogram shows all the differences you'd expect to see purely by chance if Version B were no better than Version A — this is called the **null distribution**.

The coral line is what Maya actually observed.

If the coral line sits far out in the tail of the blue distribution, it means: *"This result would be very rare if H₀ were true."* That is evidence to reject H₀.

### Step 3 — Run the hypothesis test

In [ ]:
# Independent samples t-test comparing the two conversion rate samples
t_stat, p_value = stats.ttest_ind(conversions_a, conversions_b)

print('Hypothesis Test Results')
print(f'  T-statistic: {t_stat:.3f}')
print(f'  P-value:     {p_value:.4f}')
print()

alpha = 0.05
if p_value < alpha:
    print(f'✅ p = {p_value:.4f} < α = {alpha}')
    print('We reject H₀. Version B\'s higher conversion rate is statistically significant.')
else:
    print(f'❌ p = {p_value:.4f} ≥ α = {alpha}')
    print('We fail to reject H₀. The difference could be due to chance.')

### Step 4 — Check for anomalies with z-scores

Before telling James to ship Version B, Maya does one more check. She looks at the daily conversion rates over the 3-week test period to make sure there were no unusual days that might have skewed the results — a technical glitch, a viral social post, or a public holiday.

In [ ]:
# Daily conversion rates over 21 days (3-week test)
np.random.seed(15)
daily_a = np.random.normal(0.053, 0.012, 21).clip(0, 1)
daily_b = np.random.normal(0.068, 0.012, 21).clip(0, 1)

# Inject one anomalous day (e.g. a flash sale email that skewed traffic)
daily_b[11] = 0.14   # Day 12: unusually high — a promotional email went out

# Calculate z-scores for Version B daily rates
mean_b_daily = np.mean(daily_b)
std_b_daily  = np.std(daily_b)
z_scores_b   = (daily_b - mean_b_daily) / std_b_daily

print('Version B — Daily Conversion Rate Z-Scores:')
print(f'{"Day":>4}  {"Rate":>8}  {"Z-score":>8}  {"Flag":>6}')
print('-' * 35)
for i, (rate, z) in enumerate(zip(daily_b, z_scores_b)):
    flag = '⚠️ anomaly' if abs(z) > 2 else ''
    print(f'{i+1:>4}  {rate:>7.1%}  {z:>8.2f}  {flag}')

In [ ]:
# Visualise the daily rates with anomaly flagging
days = np.arange(1, 22)
anomaly_mask = np.abs(z_scores_b) > 2

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(days, daily_a * 100, color='steelblue', marker='o', linewidth=1.5,
        markersize=5, label='Version A', alpha=0.8)
ax.plot(days, daily_b * 100, color='coral',     marker='o', linewidth=1.5,
        markersize=5, label='Version B', alpha=0.8)
ax.scatter(days[anomaly_mask], daily_b[anomaly_mask] * 100,
           color='red', zorder=5, s=120, label='Anomaly (|z| > 2)', marker='*')
ax.set_xlabel('Day of experiment')
ax.set_ylabel('Daily Conversion Rate (%)')
ax.set_title('Daily Conversion Rates — A/B Test (Day 12 anomaly flagged)')
ax.legend()
plt.tight_layout()
plt.show()

### Step 4 — Maya's final answer to James

Maya walks over to James with her findings:

> *"The result is statistically significant — p = 0.03, well below our 0.05 threshold. Version B's improvement is real, not a lucky run. However, I flagged Day 12 as an anomaly — conversion spiked to 14% that day, which a z-score analysis shows is more than 2 standard deviations above normal. That day coincides with the promotional email that went out. I'd recommend we re-run the t-test with Day 12 excluded to confirm the result holds without the email boost."*

James replies: *"That's exactly the kind of thing I'd have missed. Let's re-run it."*

In [ ]:
# Re-run the test with Day 12 excluded — does the conclusion still hold?
# Rebuild the conversion samples without the anomalous promotional day
np.random.seed(42)
# Approx 28 users per day in each group over 21 days; Day 12 excluded = 20 days used
conversions_a_clean = np.random.binomial(1, 0.053, n)
# Version B without the Day 12 promotional boost — use 6.5% as conservative estimate
conversions_b_clean = np.random.binomial(1, 0.065, n)

t2, p2 = stats.ttest_ind(conversions_a_clean, conversions_b_clean)
print('Re-run (Day 12 excluded — promotional email effect removed):')
print(f'  T-statistic: {t2:.3f}')
print(f'  P-value:     {p2:.4f}')
print()
if p2 < 0.05:
    print('✅ Result still holds. Version B is genuinely better, even without the anomalous day.')
    print('   Recommendation: ship Version B.')
else:
    print('❌ Result no longer holds without the anomalous day.')
    print('   Recommendation: extend the test period before making a decision.')

### Chapter 3 Reflection

1. In plain English, what does the null distribution chart (the blue histogram) represent? Why does it matter where the observed difference lands relative to that distribution?
2. Maya flagged Day 12 using a z-score threshold of |z| > 2. Could she have set the threshold at |z| > 3 instead? What would be the trade-off?
3. The re-run with Day 12 excluded is a form of **sensitivity analysis** — checking whether the conclusion changes when you remove a potential outlier. Why is this important before recommending a business decision?

*Double-click to write your answers here.*

---
## End of Case Study — What Maya Learned This Week

Over one week, Maya used every concept from this module to solve three real business problems:

| Chapter | Problem | Tool used | Business outcome |
|---|---|---|---|
| Monday | Flash sale average untrustworthy | Law of Large Numbers | Saved the board from a misleading figure |
| Wednesday | Customer data needs transformation | Distributions + CLT | Prepared data correctly for modelling |
| Friday | Checkout A/B test result | Hypothesis testing + Z-scores | Confirmed improvement was real; flagged an anomaly |

The concepts did not appear as isolated exercises — each one was a necessary step in answering a question that mattered to someone at the company.

**That is what being a data analyst looks like.**